# Generation Experiment: Gap Fill

This notebook runs targeted generation experiments to fill data gaps in the existing generation sweep.

**What's missing:** The original generation experiments were run in stages with non-overlapping layer/strength grids. The 8B model is complete (50 concepts, 5 trials, all layers/strengths). The 14B, 32B, and 235B models have missing (layer, strength) cells or insufficient trial counts.

**What this notebook does:** For each model, runs a clean complete sweep at depth-matched layers with 50 concepts and 5 trials, using the same settings as the original experiments (temperature 0.7, top_p 0.8, top_k 20, seed 13).

**Run order:** Set `MODEL_NAME` in the config cell, then run all cells in order. Each model produces a single `generation_gapfill.json` file.

## 1. Setup

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Skip LFS to avoid bandwidth limits — we'll generate steering vectors fresh
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/agastyasridharan/introspection.git
%cd introspection
!pip install -q accelerate transformers huggingface-hub tqdm 2>&1 | tail -3
import sys
sys.path.insert(0, "src")
print(f"Python {sys.version}")
print("Setup complete")

## 2. Configuration

Set `MODEL_NAME` and run this cell. Everything else auto-configures.

| Model | VRAM needed | Layers to run | Status |
|-------|-------------|---------------|--------|
| `Qwen/Qwen3-8B` | ~16GB | -- | **Already complete** |
| `Qwen/Qwen3-14B` | ~28GB | 6, 11, 17, 22, 28, 33, 39 | All cells missing |
| `Qwen/Qwen3-32B` | ~64GB | 9, 18, 27, 36, 45, 54, 63 | All cells missing or need trials |
| `Qwen/Qwen3-235B-A22B-Instruct-2507-FP8` | ~120GB | 13, 27, 40, 53, 66, 80, 93 | Partial |


In [ ]:
# ===== EDIT THIS =====
MODEL_NAME = "Qwen/Qwen3-14B"
# =====================

from introspection.constants import MODEL_LAYER_COUNTS

SEED = 13
STRENGTHS = [3.5, 4.0, 4.5, 5.0, 6.0]
TEMPERATURES = [0.7]
TRIALS = 5

_MODEL_DEFAULTS = {
    "Qwen/Qwen3-8B":                          {"short": "8b",      "dtype": "bfloat16", "batch": 10},
    "Qwen/Qwen3-14B":                         {"short": "14b",     "dtype": "bfloat16", "batch": 6},
    "Qwen/Qwen3-32B":                         {"short": "32b",     "dtype": "bfloat16", "batch": 3},
    "Qwen/Qwen3-235B-A22B-Instruct-2507-FP8": {"short": "235a22b", "dtype": "bfloat16", "batch": 2},
}

_defaults = _MODEL_DEFAULTS[MODEL_NAME]
DTYPE = _defaults["dtype"]
MAX_BATCH_SIZE = _defaults["batch"]

# Depth-matched layers (same as logit/mismatch experiments)
total_layers = MODEL_LAYER_COUNTS[MODEL_NAME]
_max_layer = total_layers - 1
_LAYER_FRACTIONS = [1/7, 2/7, 3/7, 4/7, 5/7, 6/7, 1.0]
LAYERS = sorted(set(round(f * _max_layer) for f in _LAYER_FRACTIONS))

DATA_DIR = f"data/qwen_{_defaults['short']}"
STEERING_VECTOR_PATH = f"{DATA_DIR}/steering_vectors.pt"
OUTPUT_PATH = f"{DATA_DIR}/generation_gapfill.json"

print(f"Model: {MODEL_NAME} ({total_layers} layers)")
print(f"Layers: {LAYERS}")
print(f"  -> layer %: {[round(l / _max_layer * 100, 1) for l in LAYERS]}")
print(f"Strengths: {STRENGTHS}")
print(f"Trials: {TRIALS}")
print(f"Batch size: {MAX_BATCH_SIZE}")
print(f"Output: {OUTPUT_PATH}")
print()
n_configs = len(LAYERS) * len(STRENGTHS) * 50
n_forward = n_configs * TRIALS / MAX_BATCH_SIZE + TRIALS  # approx
print(f"Estimated: {n_configs} configs x {TRIALS} trials")
print(f"Approx forward passes: {n_forward:.0f}")

## 2b. Load Steering Vectors

Downloads pre-computed steering vectors from HuggingFace Hub (~5 seconds).
Falls back to generating from scratch if the download fails (~10 minutes on A100).
Generation is deterministic with seed 13, so results are identical either way.

In [ ]:
from pathlib import Path
import os

sv_path = Path(STEERING_VECTOR_PATH)
if sv_path.exists() and sv_path.stat().st_size > 1000:
    print(f"Steering vectors already exist at {sv_path} ({sv_path.stat().st_size / 1e6:.1f} MB)")
    print("Skipping. Delete the file to re-download or regenerate.")
else:
    try:
        from huggingface_hub import hf_hub_download
        print(f"Downloading steering vectors for {MODEL_NAME}...")
        sv_path.parent.mkdir(parents=True, exist_ok=True)
        hf_hub_download(
            repo_id="agastyasridharan/introspection-steering-vectors",
            filename=f"qwen_{_defaults['short']}/steering_vectors.pt",
            local_dir="data",
            repo_type="dataset",
        )
        print(f"Downloaded! ({sv_path.stat().st_size / 1e6:.1f} MB)")
    except Exception as e:
        print(f"Download failed ({e}), generating from scratch...")
        from introspection.generate_steering_vectors import run_experiment
        os.makedirs(sv_path.parent, exist_ok=True)
        run_experiment(
            model_name=MODEL_NAME,
            dtype_name=DTYPE,
            output_path=sv_path,
            concept_count=50,
            seed=SEED,
        )
        print(f"Generated! ({sv_path.stat().st_size / 1e6:.1f} MB)")


## 3. Run Generation Sweep

In [ ]:
from pathlib import Path
from introspection.steer import (
    load_steering_vectors, prepare_prompt, steer as run_steer,
    PROMPT_MESSAGES,
)
from introspection.types import ExperimentArgs
from introspection.utils import load_model, resolve_torch_dtype
import json

# Load model
print(f"Loading {MODEL_NAME}...")
tokenizer, model = load_model(
    model_name=MODEL_NAME,
    dtype=resolve_torch_dtype(DTYPE),
    disable_cache=False,
    set_pad_token_to_eos=True,
)
print(f"Model loaded on {model.device}")

# Load steering vectors
steering_vectors = load_steering_vectors(Path(STEERING_VECTOR_PATH))
concept_names = sorted(steering_vectors.keys())
print(f"Loaded {len(concept_names)} concepts")

# Prepare prompt
template_prompt = prepare_prompt(tokenizer, model.device)

# Build args
args = ExperimentArgs(
    model_name=MODEL_NAME,
    dtype_name=DTYPE,
    steering_vector_path=Path(STEERING_VECTOR_PATH),
    concepts=None,
    layers=LAYERS,
    strengths=STRENGTHS,
    json_path=Path(OUTPUT_PATH),
    temperatures=TEMPERATURES,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    trials=TRIALS,
    max_new_tokens=200,
    do_sample=True,
    seed=SEED,
    debug_residual=False,
    max_batch_size=MAX_BATCH_SIZE,
)

# Run
print(f"\n=== Running {len(concept_names)} concepts x {len(LAYERS)} layers x {len(STRENGTHS)} strengths x {TRIALS} trials ===")
records = run_steer(
    args=args,
    concept_names=concept_names,
    all_steering_vectors=steering_vectors,
    tokenizer=tokenizer,
    model=model,
    template_prompt=template_prompt,
)

# Save
output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)
experiment_summary = {
    "model_name": MODEL_NAME,
    "steering_vector_path": str(args.steering_vector_path),
    "dtype": DTYPE,
    "prompt": {
        "messages": PROMPT_MESSAGES,
        "formatted": template_prompt.formatted_prompt,
        "injection_index": template_prompt.injection_index,
    },
    "settings": {
        "strengths": STRENGTHS,
        "temperatures": TEMPERATURES,
        "top_p": args.top_p,
        "top_k": args.top_k,
        "min_p": args.min_p,
        "max_new_tokens": args.max_new_tokens,
        "trials": TRIALS,
        "do_sample": args.do_sample,
        "seed": SEED,
        "layers": LAYERS,
        "concepts_requested": None,
        "max_batch_size": MAX_BATCH_SIZE,
    },
    "concepts_evaluated": concept_names,
    "results": records,
}
with output_path.open("w", encoding="utf-8") as f:
    json.dump(experiment_summary, f, ensure_ascii=False, indent=2)
    f.write("\n")

print(f"\nSaved {len(records)} records to {output_path}")

## 4. Quick Sanity Check

In [ ]:
import json, random

with open(OUTPUT_PATH) as f:
    data = json.load(f)

results = data["results"]
print(f"Total records: {len(results)}")
print(f"Concepts: {len(data['concepts_evaluated'])}")
print(f"Layers: {sorted(set(str(r['layers']) for r in results))}")
print(f"Strengths: {sorted(set(r['strength'] for r in results))}")
print(f"Trials: {sorted(set(r['trial'] for r in results))}")

# Check control responses
controls = set(r['control'] for r in results)
print(f"\nUnique control responses: {len(controls)}")
for c in sorted(controls):
    is_deny = "don't" in c.lower() or 'not' in c.lower()[:50]
    label = 'DENY' if is_deny else 'CLAIM'
    print(f"  [{label}] {c[:120]}")

print("\n" + "="*80)
for r in random.sample(results, min(3, len(results))):
    print(f"\nConcept: {r['concept']} | Layer: {r['layers']} | Strength: {r['strength']} | Trial: {r['trial']}")
    print(f"  Control:      {r['control'][:150]}")
    print(f"  Intervention: {r['intervention'][:150]}")

## 5. Download Results

In [ ]:
from google.colab import files
import os

if os.path.exists(OUTPUT_PATH):
    size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
    print(f"Downloading {OUTPUT_PATH} ({size_mb:.1f} MB)...")
    files.download(OUTPUT_PATH)
else:
    print(f"No file at {OUTPUT_PATH}")